# 假阳性去除改进测试（v2）

**核心改动（仅一处）：** 用 `save_matched_cells_v2` 替换原 `save_matched_cells`，增加两层细胞级筛选：
- **层1（门槛层）**：与任意恶性参考簇的最大相似度 ≥ `CELL_THRESHOLD_TARGET`
- **层2（专一层）**：最佳恶性簇相似度 ≥ 所有良性参考簇相似度 × `CELL_THRESHOLD_RATIO`

其余逻辑与 `pipeline.py` 完全一致。

**调参流程：** 修改 Cell 1 → 运行 Cell 1~7 → 查看 Cell 8~9 → 用 Cell 10~11 找最优参数 → 只改 Cell 1 重跑 Cell 7~11

## Cell 1：配置区 —— 只需修改这里

In [ ]:
import os, sys

PROJECT_ROOT         = r"E:\limr\Soft\shilab-cluster-algorithm"
MALIGNANT_CELLS_DIR  = r"H:/limr/project/urine/rawdata/classification_model/UR_SingleCell/malignant"
BENIGN_CELLS_DIR     = r"H:/limr/project/urine/rawdata/classification_model/UR_SingleCell/benign"
POSITIVE_FOLDER_PATH = r"H:\limr\project\urine\result\binary_infer\sample31\single_cell\test\pos"
NEGATIVE_FOLDER_PATH = r"H:\limr\project\urine\result\binary_infer\sample31\single_cell\test\neg"
MODEL_PATH  = r"H:\limr\project\urine\result\classification_model\UR_SingleCell_seed300_v3\train_val_models\DenseNet161\densenet161_fold_4.pth"
MODEL_TYPE  = "DenseNet161"
MEAN        = [0.5918, 0.5688, 0.7202]
STD         = [0.2469, 0.2356, 0.1270]
BASE_SAVE_DIR = r"G:\04PhD\03Study\05SelfLearn\results\clustering\test_v2"

FEATURE_LAYER    = "penultimate"
REFERENCE_K      = 10
MAX_CANDIDATE_K  = 20
BATCH_SIZE       = 16
NUM_WORKERS      = 0
CELL_SIZE_WEIGHT = 1.0
PCA_DIM          = 32

MALIGNANT_REF_CLUSTERS = [8, 7, 1]

MATCHING_RULES = [
    {"target": 8, "avoid": [7, 1], "threshold_target": 0.5, "threshold_ratio": 3},
    {"target": 7, "avoid": [8, 1], "threshold_target": 0.5, "threshold_ratio": 3},
    {"target": 1, "avoid": [8, 7], "threshold_target": 0.5, "threshold_ratio": 3},
]

# ---- 细胞级筛选参数（本次改进核心，重点调这里）----
CELL_THRESHOLD_TARGET = 0.5   # 层1：与恶性簇最低相似度（建议范围 0.4~0.65）
CELL_THRESHOLD_RATIO  = 3.0   # 层2：恶性相似度 / 良性簇最大相似度 的最低倍数（建议范围 1.5~5.0）

SAVE_REJECTED_CELLS = True    # True=保存被拒绝的细胞，方便人工核查

print("配置加载完成！")
print(f"  模型：{MODEL_TYPE}  恶性簇：{MALIGNANT_REF_CLUSTERS}")
print(f"  层1阈值：{CELL_THRESHOLD_TARGET}  层2比例：{CELL_THRESHOLD_RATIO}")
print(f"  输出目录：{BASE_SAVE_DIR}")


## Cell 2：导入库和模块

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shutil
import warnings
warnings.filterwarnings("ignore")
from tqdm import tqdm
from torch.utils.data import DataLoader
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

sys.path.insert(0, PROJECT_ROOT)

from cross_cluster_matching.models.models import get_model_and_transform
from cross_cluster_matching.data.data_utils import (
    collect_reference_data, collect_image_paths, CellImageDataset, extract_patient_id
)
from cross_cluster_matching.models.feature_extractor import extract_features_with_cell_size
from cross_cluster_matching.clustering.K_optimizer import optimize_k_for_candidate_clustering
from cross_cluster_matching.clustering.calculate_consensus_score import calculate_consensus_score
from cross_cluster_matching.clustering.matching_rule_application import (
    identify_matching_clusters, check_matching_rules
)
from cross_cluster_matching.visualization.visualization import (
    count_non_background_pixels,
    visualize_all_clusters_without_images,
    visualize_all_clusters_without_images_1,
    improved_visualize_clusters_with_images,
    visualize_matching_clusters,
    analyze_cluster_composition,
    plot_similarity_heatmap,
    visualize_filtered_clusters,
)

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(BASE_SAVE_DIR, exist_ok=True)
print(f"所有模块导入成功，设备：{device}")


## Cell 3：定义改进的细胞级筛选函数（核心改动）

> 此 Cell 只需运行一次，不需要修改。

In [ ]:
def check_cell_rules(cell_sim_vec, target_ref_indices, threshold_target, threshold_ratio):
    n_ref = len(cell_sim_vec)
    benign_indices = [i for i in range(n_ref) if i not in target_ref_indices]
    mal_sims     = cell_sim_vec[target_ref_indices]
    best_mal_sim = float(mal_sims.max())
    best_mal_idx = target_ref_indices[int(mal_sims.argmax())]
    # 层1
    if best_mal_sim < threshold_target:
        return False, {"rejected_layer": 1,
                       "reason": f"恶性相似度{best_mal_sim:.3f} < 阈值{threshold_target}",
                       "best_mal_sim": best_mal_sim, "best_mal_idx": best_mal_idx, "max_ben_sim": None}
    # 层2
    if benign_indices:
        max_ben_sim = float(cell_sim_vec[benign_indices].max())
        limit = best_mal_sim / threshold_ratio
        if max_ben_sim >= limit:
            return False, {"rejected_layer": 2,
                           "reason": f"良性相似度{max_ben_sim:.3f} >= limit {limit:.3f}",
                           "best_mal_sim": best_mal_sim, "best_mal_idx": best_mal_idx, "max_ben_sim": max_ben_sim}
    else:
        max_ben_sim = 0.0
    return True, {"rejected_layer": None, "reason": "通过",
                  "best_mal_sim": best_mal_sim, "best_mal_idx": best_mal_idx, "max_ben_sim": max_ben_sim}


def save_matched_cells_v2(indices, image_paths, labels, cell_sim_matrix,
                          rule_type, save_dirs, target_ref_indices,
                          threshold_target=0.5, threshold_ratio=3.0,
                          save_rejected=False, rejected_dir=None):
    count, processed_indices = 0, []
    stats = {"total": len(indices), "passed": 0,
             "rejected_layer1": 0, "rejected_layer2": 0, "details": []}
    if save_rejected and rejected_dir:
        os.makedirs(os.path.join(rejected_dir, "layer1"), exist_ok=True)
        os.makedirs(os.path.join(rejected_dir, "layer2"), exist_ok=True)
    for idx in indices:
        passed, info = check_cell_rules(
            cell_sim_matrix[idx], target_ref_indices, threshold_target, threshold_ratio
        )
        src_path      = image_paths[idx]
        filename      = os.path.basename(src_path)
        cluster_label = labels[idx]
        if passed:
            prefix   = f"{rule_type}_cluster{cluster_label}_ref{info[chr(39)+chr(98)+chr(101)+chr(115)+chr(116)+chr(95)+chr(109)+chr(97)+chr(108)+chr(95)+chr(105)+chr(100)+chr(39)]}"
            new_name = f"{prefix}_{filename}"
            shutil.copy(src_path, os.path.join(save_dirs["rule_dir"],  new_name))
            shutil.copy(src_path, os.path.join(save_dirs["total_dir"], new_name))
            count += 1
            processed_indices.append(idx)
            stats["passed"] += 1
        else:
            layer = info["rejected_layer"]
            stats[f"rejected_layer{layer}"] += 1
            if save_rejected and rejected_dir:
                sub = f"layer{layer}"
                shutil.copy(src_path, os.path.join(rejected_dir, sub, f"rej_c{cluster_label}_{filename}"))
        stats["details"].append({
            "idx": idx, "passed": passed,
            "best_mal_sim": info["best_mal_sim"], "best_mal_idx": info["best_mal_idx"],
            "max_ben_sim": info["max_ben_sim"],
            "rejected_layer": info["rejected_layer"], "reason": info["reason"],
        })
    return count, processed_indices, stats


print("改进的细胞级筛选函数定义完成")
print(f"  层1阈值：CELL_THRESHOLD_TARGET = {CELL_THRESHOLD_TARGET}")
print(f"  层2比例：CELL_THRESHOLD_RATIO  = {CELL_THRESHOLD_RATIO}")


## Cell 4：加载模型

In [ ]:
print("加载模型...")
model, transform, _ = get_model_and_transform(
    MODEL_TYPE, device, MODEL_PATH, mean=MEAN, std=STD
)
model.eval()
print(f"模型加载完成：{MODEL_TYPE}")


## Cell 5：处理参考域（与 pipeline.py 完全一致）

In [ ]:
print("======= 处理参考细胞 =======")
ref_paths, ref_sources = collect_reference_data(MALIGNANT_CELLS_DIR, BENIGN_CELLS_DIR)
print(f"参考域总数：{len(ref_paths)}")
print(f"标签分布：{pd.Series(ref_sources).value_counts().to_dict()}")

ref_sizes = np.log1p([
    count_non_background_pixels(p) for p in tqdm(ref_paths, desc="Ref Sizes")
])
ref_loader = DataLoader(
    CellImageDataset(ref_paths, transform=transform, source_labels=ref_sources),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)
ref_feats, ref_sources, ref_paths = extract_features_with_cell_size(
    model, ref_loader, device, ref_sizes, FEATURE_LAYER, CELL_SIZE_WEIGHT
)
print(f"参考域特征维度：{ref_feats.shape}")

pca = PCA(n_components=min(PCA_DIM, ref_feats.shape[0], ref_feats.shape[1]))
ref_norm = normalize(pca.fit_transform(ref_feats))
kmeans_ref = KMeans(n_clusters=REFERENCE_K, random_state=SEED, n_init=10).fit(ref_norm)
ref_results = {
    "features"       : ref_norm,
    "labels"         : kmeans_ref.labels_,
    "source_labels"  : ref_sources,
    "image_paths"    : ref_paths,
    "cluster_centers": kmeans_ref.cluster_centers_,
    "n_clusters"     : REFERENCE_K,
    "pca"            : pca,
}
print(f"参考域聚类完成，K={REFERENCE_K}")

ref_vis_dir = os.path.join(BASE_SAVE_DIR, f"reference_visualization_pca{PCA_DIM}")
os.makedirs(ref_vis_dir, exist_ok=True)
reference_umap = visualize_all_clusters_without_images(
    ref_results["features"], ref_results["labels"],
    ref_results["source_labels"], "Reference", ref_vis_dir
)
reference_umap_1 = visualize_all_clusters_without_images_1(
    ref_results["features"], ref_results["labels"],
    ref_results["source_labels"], "Reference", ref_vis_dir
)
improved_visualize_clusters_with_images(
    ref_results["features"], ref_results["labels"],
    ref_results["source_labels"], ref_results["image_paths"],
    "Reference", ref_vis_dir, ref_sizes
)
analyze_cluster_composition(
    ref_results["labels"], ref_results["source_labels"],
    ref_results["image_paths"], "Reference", ref_vis_dir
)

# 核查恶性簇索引
print("\n参考域各簇构成（核查 MALIGNANT_REF_CLUSTERS 是否正确）：")
print(f"{chr(31807):<6}{chr(24635)chr(25968):<8}{chr(24076)chr(24615)chr(25968):<10}{chr(33391)chr(24615)chr(25968):<10}{chr(24076)chr(24615)chr(27604)chr(20363):<10}{chr(26631)chr(35760)}")
print("-" * 55)
for c in range(REFERENCE_K):
    mask  = ref_results["labels"] == c
    srcs  = np.array(ref_results["source_labels"])[mask]
    mal   = int(np.sum((srcs == 5) | (srcs == 1)))
    ben   = int(np.sum((srcs == 4) | (srcs == 0)))
    tot   = int(mask.sum())
    ratio = mal / tot if tot > 0 else 0
    flag  = "*** Malignant" if c in MALIGNANT_REF_CLUSTERS else ""
    print(f"{c:<6}{tot:<8}{mal:<10}{ben:<10}{ratio:<10.2%}{flag}")


## Cell 6：收集候选域样本列表

In [ ]:
patient_folders = []
if os.path.exists(POSITIVE_FOLDER_PATH):
    patient_folders.extend([
        (e.path, "positive") for e in os.scandir(POSITIVE_FOLDER_PATH) if e.is_dir()
    ])
if os.path.exists(NEGATIVE_FOLDER_PATH):
    patient_folders.extend([
        (e.path, "negative") for e in os.scandir(NEGATIVE_FOLDER_PATH) if e.is_dir()
    ])
n_pos = sum(1 for _, s in patient_folders if s == "positive")
n_neg = sum(1 for _, s in patient_folders if s == "negative")
print(f"找到候选样本：阳性={n_pos}，阴性={n_neg}，共{len(patient_folders)}个")


## Cell 7：处理所有候选样本

> 每处理完一个样本立即打印细胞级筛选统计。  
> 调参后只需修改 Cell 1，然后重跑本 Cell。

In [ ]:
n_rules = len(MALIGNANT_REF_CLUSTERS)
results_stats         = []
all_sample_cell_stats = {}

for folder, status in patient_folders:
    pid      = extract_patient_id(folder)
    save_dir = os.path.join(BASE_SAVE_DIR, pid)
    os.makedirs(save_dir, exist_ok=True)

    cand_paths = collect_image_paths(os.path.join(folder, "malignant_images"))
    if not cand_paths:
        print(f"  {pid}: malignant_images empty, skip")
        continue

    print(f"\n{"="*60}")
    print(f"Processing: {pid}  [{status}]  cells={len(cand_paths)}")
    print(f"{"="*60}")

    cand_sources = [0] * len(cand_paths)
    cand_sizes   = np.log1p([count_non_background_pixels(p) for p in cand_paths])

    cand_loader = DataLoader(
        CellImageDataset(cand_paths, transform=transform, source_labels=cand_sources),
        batch_size=BATCH_SIZE, shuffle=False
    )
    cand_feats, _, cand_paths = extract_features_with_cell_size(
        model, cand_loader, device, cand_sizes, FEATURE_LAYER, CELL_SIZE_WEIGHT
    )
    cand_norm = normalize(pca.transform(cand_feats))

    matched_counts = [0] * n_rules
    dirs = {f"rule{i+1}": os.path.join(save_dir, f"rule{i+1}_matched_cells") for i in range(n_rules)}
    dirs["total"] = os.path.join(save_dir, "matched_malignant_cells")
    for d in dirs.values():
        os.makedirs(d, exist_ok=True)
    rejected_base = os.path.join(save_dir, "rejected_cells")

    cell_sims = cosine_similarity(cand_norm, ref_results["cluster_centers"])
    sample_stats_list = []

    if len(cand_norm) >= 32:
        cand_results, best_k = optimize_k_for_candidate_clustering(
            cand_norm, cand_sources, cand_paths,
            ref_results, save_dir, pid, MAX_CANDIDATE_K, MALIGNANT_REF_CLUSTERS
        )
        if not cand_results:
            continue
        print(f"  Best K: {best_k}")

        pd.DataFrame({"labels": cand_results["labels"], "image_paths": cand_results["image_paths"]})\
          .to_csv(os.path.join(save_dir, "cluster_detail.csv"), index=False, encoding="utf-8-sig")

        rule_result = identify_matching_clusters(
            cand_results["cluster_centers"], ref_results["cluster_centers"], MATCHING_RULES
        )
        similarity_matrix  = rule_result[-1]
        rule_clusters_list = list(rule_result[:-1])

        print("  Cluster-level match:")
        for i, rc in enumerate(rule_clusters_list):
            print(f"    rule{i+1} (target=R{MATCHING_RULES[i][chr(116)+chr(97)+chr(114)+chr(103)+chr(101)+chr(116)}]): matched candidate clusters {rc}")

        print(f"\n  [Cell-level filter] threshold_target={CELL_THRESHOLD_TARGET}, threshold_ratio={CELL_THRESHOLD_RATIO}")
        for i, rule_clusters in enumerate(rule_clusters_list):
            rule_name = f"rule{i+1}"
            indices   = [j for j, lbl in enumerate(cand_results["labels"]) if lbl in rule_clusters]
            if not indices:
                print(f"  {rule_name}: no matched cluster, skip")
                continue
            rej_dir = os.path.join(rejected_base, rule_name) if SAVE_REJECTED_CELLS else None
            count, _, stats = save_matched_cells_v2(
                indices, cand_paths, cand_results["labels"], cell_sims,
                rule_name, {"rule_dir": dirs[rule_name], "total_dir": dirs["total"]},
                target_ref_indices=MALIGNANT_REF_CLUSTERS,
                threshold_target=CELL_THRESHOLD_TARGET,
                threshold_ratio=CELL_THRESHOLD_RATIO,
                save_rejected=SAVE_REJECTED_CELLS, rejected_dir=rej_dir,
            )
            matched_counts[i] = count
            sample_stats_list.append({"rule": rule_name, **stats})
            print(f"  {rule_name}: input={stats[chr(116)+chr(111)+chr(116)+chr(97)+chr(108)]} kept={stats[chr(112)+chr(97)+chr(115)+chr(115)+chr(101)+chr(100)]} | L1_rej={stats[chr(114)+chr(101)+chr(106)+chr(101)+chr(99)+chr(116)+chr(101)+chr(100)+chr(95)+chr(108)+chr(97)+chr(121)+chr(101)+chr(114)+chr(49)]} L2_rej={stats[chr(114)+chr(101)+chr(106)+chr(101)+chr(99)+chr(116)+chr(101)+chr(100)+chr(95)+chr(108)+chr(97)+chr(121)+chr(101)+chr(114)+chr(50)]}")

        matching_pairs = []
        for rule, rcs in zip(MATCHING_RULES, rule_clusters_list):
            for ci in rcs:
                matching_pairs.append((rule["target"], ci, similarity_matrix[ci, rule["target"]]))
        rule_matched_clusters = {f"rule{i+1}": c for i, c in enumerate(rule_clusters_list)}

        calculate_consensus_score(ref_results, cand_results, matching_pairs, save_dir)
        visualize_matching_clusters(ref_results, cand_results, matching_pairs, save_dir, reference_umap, rule_matched_clusters)
        visualize_all_clusters_without_images(cand_results["features"], cand_results["labels"], cand_results["source_labels"], pid, save_dir, reference_umap)
        improved_visualize_clusters_with_images(cand_results["features"], cand_results["labels"], cand_results["source_labels"], cand_paths, pid, save_dir, cand_sizes)
        visualize_filtered_clusters(ref_results, cand_results, matching_pairs, save_dir, rule_matched_clusters=rule_matched_clusters, target_ref_indices=MALIGNANT_REF_CLUSTERS)
        plot_similarity_heatmap(
            similarity_matrix.T,
            [f"C{i}" for i in range(cand_results["n_clusters"])],
            [f"R{i}" for i in range(ref_results["n_clusters"])],
            f"{pid} Similarity Matrix",
            os.path.join(save_dir, "similarity_heatmap.png")
        )

    else:
        print(f"  cells({len(cand_norm)}) < 32, single-cell mode")
        processed = set()
        for i, rule in enumerate(MATCHING_RULES):
            rule_name = f"rule{i+1}"
            indices = [
                j for j, s in enumerate(cell_sims)
                if j not in processed and check_matching_rules(
                    s, ref_idx_target=rule["target"], ref_idx_avoid=rule["avoid"],
                    threshold_target=rule["threshold_target"],
                    threshold_avoid_others=rule.get("threshold_avoid_others"),
                    threshold_ratio=rule.get("threshold_ratio"),
                )
            ]
            rej_dir = os.path.join(rejected_base, rule_name) if SAVE_REJECTED_CELLS else None
            count, newly_processed, stats = save_matched_cells_v2(
                indices, cand_paths, [0]*len(cand_paths), cell_sims,
                rule_name, {"rule_dir": dirs[rule_name], "total_dir": dirs["total"]},
                target_ref_indices=MALIGNANT_REF_CLUSTERS,
                threshold_target=CELL_THRESHOLD_TARGET,
                threshold_ratio=CELL_THRESHOLD_RATIO,
                save_rejected=SAVE_REJECTED_CELLS, rejected_dir=rej_dir,
            )
            matched_counts[i] = count
            processed.update(newly_processed)
            sample_stats_list.append({"rule": rule_name, **stats})
            print(f"  {rule_name}: input={stats[chr(116)+chr(111)+chr(116)+chr(97)+chr(108)]} kept={stats[chr(112)+chr(97)+chr(115)+chr(115)+chr(101)+chr(100)]} | L1_rej={stats[chr(114)+chr(101)+chr(106)+chr(101)+chr(99)+chr(116)+chr(101)+chr(100)+chr(95)+chr(108)+chr(97)+chr(121)+chr(101)+chr(114)+chr(49)]} L2_rej={stats[chr(114)+chr(101)+chr(106)+chr(101)+chr(99)+chr(116)+chr(101)+chr(100)+chr(95)+chr(108)+chr(97)+chr(121)+chr(101)+chr(114)+chr(50)]}")
        plot_similarity_heatmap(
            cell_sims.T,
            [f"Cell{i}" for i in range(len(cell_sims))],
            [f"R{i}" for i in range(ref_results["n_clusters"])],
            f"Patient {pid} Cell Similarity Matrix",
            os.path.join(save_dir, "Cell_similarity_heatmap.png")
        )

    total_input   = len(cand_paths)
    total_kept    = sum(matched_counts)
    total_removed = total_input - total_kept
    print(f"\n  {pid} done: input={total_input} kept={total_kept}({total_kept/total_input*100:.1f}%) removed={total_removed}({total_removed/total_input*100:.1f}%)")
    results_stats.append({
        "Patient_ID": pid, "Status": status, "Total_Input": total_input,
        **{f"Rule{i+1}": matched_counts[i] for i in range(n_rules)},
        "Total_Kept": total_kept, "Total_Removed": total_removed,
        "Remove_Rate": f"{total_removed/total_input*100:.1f}%" if total_input > 0 else "N/A",
    })
    all_sample_cell_stats[pid] = sample_stats_list

print(f"\n{"="*60}\nAll specimens done!")


## Cell 8：汇总统计表

In [ ]:
stats_df = pd.DataFrame(results_stats).sort_values(["Status", "Patient_ID"])
excel_path = os.path.join(BASE_SAVE_DIR, "stats_v2.xlsx")
stats_df.to_excel(excel_path, index=False)
print("\n[Summary Table]")
print(stats_df.to_string(index=False))
print(f"\nSaved: {excel_path}")
print("\n[Group Statistics]")
for status in ["negative", "positive"]:
    sub = stats_df[stats_df["Status"] == status]
    if len(sub) == 0: continue
    avg_remove = sub["Total_Removed"].sum() / sub["Total_Input"].sum() * 100
    print(f"  {status} ({len(sub)} samples): avg remove rate = {avg_remove:.1f}%")


## Cell 9：去除率可视化

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle(f"Cell-level Filter Remove Rate (L1={CELL_THRESHOLD_TARGET}, L2={CELL_THRESHOLD_RATIO})",
             fontsize=13, fontweight="bold")
for ax, status, title, color in zip(
    axes,
    ["negative", "positive"],
    ["Negative (expect high remove rate)", "Positive (expect moderate remove rate)"],
    ["#C82423", "#2471A3"]
):
    sub = stats_df[stats_df["Status"] == status]
    if len(sub) == 0:
        ax.set_visible(False)
        continue
    remove_rates = sub["Total_Removed"] / sub["Total_Input"] * 100
    bars = ax.bar(sub["Patient_ID"], remove_rates, color=color, alpha=0.8, edgecolor="white")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylabel("Remove Rate (%)")
    ax.set_ylim(0, 110)
    ax.tick_params(axis="x", rotation=45)
    ax.grid(True, alpha=0.3, axis="y")
    for bar, rate in zip(bars, remove_rates):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f"{rate:.1f}%", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(BASE_SAVE_DIR, "remove_rate_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()


## Cell 10：参数敏感性分析

> 不重新跑模型，直接用 Cell 7 保存的相似度数据，快速扫描所有参数组合。

In [ ]:
print("Parameter sensitivity analysis (no re-inference needed)\n")
param_rows = []
for thr in [0.40, 0.45, 0.50, 0.55, 0.60, 0.65]:
    for ratio in [2.0, 2.5, 3.0, 3.5, 4.0, 5.0]:
        neg_rr, pos_rr = [], []
        for pid, ssl in all_sample_cell_stats.items():
            st  = next(r["Status"]     for r in results_stats if r["Patient_ID"] == pid)
            tot = next(r["Total_Input"] for r in results_stats if r["Patient_ID"] == pid)
            kept = sum(
                1 for rs in ssl for d in rs.get("details", [])
                if d["best_mal_sim"] >= thr
                and (d["max_ben_sim"] is None or d["max_ben_sim"] < d["best_mal_sim"] / ratio)
            )
            rr = (tot - kept) / tot if tot > 0 else 0
            (neg_rr if st == "negative" else pos_rr).append(rr)
        neg_avg = np.mean(neg_rr) * 100 if neg_rr else 0
        pos_avg = np.mean(pos_rr) * 100 if pos_rr else 0
        param_rows.append({"threshold_target": thr, "threshold_ratio": ratio,
                           "neg_remove%": round(neg_avg,1), "pos_remove%": round(pos_avg,1),
                           "advantage": round(neg_avg - pos_avg, 1)})

param_df = pd.DataFrame(param_rows).sort_values("advantage", ascending=False)
param_df.to_csv(os.path.join(BASE_SAVE_DIR, "param_sensitivity.csv"), index=False, encoding="utf-8-sig")
print("Top 10 (advantage = neg_remove% - pos_remove%, higher is better):")
print(param_df.head(10).to_string(index=False))
print(f"\nCurrent: threshold_target={CELL_THRESHOLD_TARGET}, threshold_ratio={CELL_THRESHOLD_RATIO}")
cur = param_df[(param_df["threshold_target"]==CELL_THRESHOLD_TARGET) & (param_df["threshold_ratio"]==CELL_THRESHOLD_RATIO)]
if len(cur): print(cur.to_string(index=False))


## Cell 11：参数敏感性热力图

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle("Parameter Sensitivity Heatmap", fontsize=14, fontweight="bold")
thrs = sorted(param_df["threshold_target"].unique())
rats = sorted(param_df["threshold_ratio"].unique())
for ax, col, title, cmap in zip(
    axes,
    ["neg_remove%", "pos_remove%", "advantage"],
    ["Neg Remove% (higher=better)", "Pos Remove% (lower=better)", "Advantage=Neg-Pos (higher=better)"],
    ["Reds", "Blues_r", "RdYlGn"]
):
    pivot = param_df.pivot(index="threshold_ratio", columns="threshold_target", values=col)
    sns.heatmap(pivot, annot=True, fmt=".1f", cmap=cmap, ax=ax, cbar_kws={"label": col})
    ax.set_title(title, fontsize=11, fontweight="bold")
    if CELL_THRESHOLD_TARGET in thrs and CELL_THRESHOLD_RATIO in rats:
        ax.add_patch(plt.Rectangle(
            (thrs.index(CELL_THRESHOLD_TARGET), rats.index(CELL_THRESHOLD_RATIO)),
            1, 1, fill=False, edgecolor="black", lw=3
        ))
plt.tight_layout()
plt.savefig(os.path.join(BASE_SAVE_DIR, "param_sensitivity_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Black box = current setting. Adjust Cell 1 params and re-run Cell 7~11.")
